# 02 — Log simulation from sensor anomalies

## What `sensor_df` is

`generate_logs(sensor_df)` expects a **pandas DataFrame** with (at least) these columns:

- `timestamp` — aligned to the sensor time series  
- `building_id` — building name  
- `anomaly_score` — e.g. reconstruction error  
- `anomaly_flag` — `0` / `1`  

Those rows are produced by the **sensor testing pipeline** in `scripts/Model_Testing.py`, function `export_sensor_outputs`, which writes:

`data/processed/sensor_outputs/<BuildingName>_anomalies.csv`

## Run order

1. Train models and run **`python scripts/Model_Testing.py`** (from the project root, with `config.BUILDINGS` and trained `*_best.h5` models) so `*_anomalies.csv` files exist.  
2. Run the next code cell in this notebook to generate `*_logs.csv` under `data/processed/logs_raw/`.

Only buildings listed in **`src.config.BUILDINGS`** are read from `sensor_outputs/` (extra `*_anomalies.csv` files are ignored so the log pipeline stays aligned with the sensor study).

In [20]:
import os
import sys
import importlib

import pandas as pd

# Notebook lives in notebooks/ — project root is one level up
NOTEBOOK_DIR = os.path.abspath(os.getcwd())
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
SCRIPTS = os.path.join(PROJECT_ROOT, "scripts")
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)

# Reload so edits to scripts/log_pipeline/log_generator.py apply without restarting the kernel
import log_pipeline.log_generator as _log_gen
importlib.reload(_log_gen)
generate_logs = _log_gen.generate_logs

SENSOR_OUT = os.path.join(PROJECT_ROOT, "data", "processed", "sensor_outputs")
LOGS_RAW = os.path.join(PROJECT_ROOT, "data", "processed", "logs_raw")
os.makedirs(LOGS_RAW, exist_ok=True)

# Align with sensor LSTM cohort (src.config.BUILDINGS) — only these get *_logs.csv here.
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from src.config import BUILDINGS

SENSOR_STUDY_BUILDINGS = list(BUILDINGS)

print("Project root:", PROJECT_ROOT)
print("Sensor anomalies:", SENSOR_OUT)
print("Log output:", LOGS_RAW)

Project root: D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection
Sensor anomalies: D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\sensor_outputs
Log output: D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\logs_raw


In [22]:
if not os.path.isdir(SENSOR_OUT):
    raise FileNotFoundError(
        f"Sensor output folder missing: {SENSOR_OUT}\n"
        "Run the sensor pipeline first: from project root, execute `python scripts/Model_Testing.py` "
        "(requires trained models under models/)."
    )

files = sorted(f for f in os.listdir(SENSOR_OUT) if f.endswith("_anomalies.csv"))
cohort = set(SENSOR_STUDY_BUILDINGS)
files = [f for f in files if f.replace("_anomalies.csv", "") in cohort]
if not files:
    raise FileNotFoundError(
        f"No *_anomalies.csv for sensor cohort {sorted(cohort)} under {SENSOR_OUT}. "
        "Run scripts/Model_Testing.py for those buildings first."
    )

missing_exports = sorted(cohort - {f.replace("_anomalies.csv", "") for f in files})
if missing_exports:
    print("WARNING: cohort buildings without anomaly CSV (skipped):", missing_exports)

print("Found sensor files (cohort only):", files)

GEN_CONFIG = {
    "anomaly_probability": 0.35,
    "burst_frequency": 0.55,
    "burst_length": (1, 2),
    "min_anomaly_rate": 0.08,
    "target_anomaly_rate": 0.10,
    "max_anomaly_rate": 0.20,
    "partial_sequence_prob": 0.03,
    "partial_sequence_max_events": 2,
    "ambiguous_normal_prob": 0.01,
    "spurious_anomaly_link_flip_prob": 0.0,
}


def classify_anomaly(df: pd.DataFrame) -> pd.DataFrame:
    """Assign anomaly_type per flagged row (used by generate_logs for log bursts)."""
    df = df.copy()
    df["rolling_mean"] = df["anomaly_score"].rolling(5).mean()
    df["rolling_std"] = df["anomaly_score"].rolling(5).std()
    df["anomaly_type"] = "normal"

    for i in range(1, len(df)):
        if df.loc[i, "anomaly_flag"] == 0:
            continue
        current = df.loc[i, "anomaly_score"]
        prev = df.loc[i - 1, "anomaly_score"]

        if current > prev * 1.5:
            df.loc[i, "anomaly_type"] = "spike"
        elif current < prev * 0.5:
            df.loc[i, "anomaly_type"] = "drop"
        elif df.loc[i, "rolling_std"] < 0.01:
            df.loc[i, "anomaly_type"] = "flatline"
        else:
            df.loc[i, "anomaly_type"] = "drift"

    return df


summary_rows = []
for file in files:
    print(f"\nProcessing {file}...")
    path = os.path.join(SENSOR_OUT, file)
    sensor_df = pd.read_csv(path)
    sensor_df["timestamp"] = pd.to_datetime(sensor_df["timestamp"])

    building_name = file.replace("_anomalies.csv", "")
    sensor_df["building_id"] = building_name

    sensor_df = classify_anomaly(sensor_df)

    logs = generate_logs(sensor_df, **GEN_CONFIG)
    log_df = pd.DataFrame(logs)

    save_path = os.path.join(LOGS_RAW, f"{building_name}_logs.csv")
    log_df.to_csv(save_path, index=False)

    anomaly_rate = float(log_df["anomaly_link"].mean()) if len(log_df) else 0.0
    n_anom = int((log_df["anomaly_link"] == 1).sum())
    n_total = int(len(log_df))
    n_burst_events = int(
        log_df[log_df["anomaly_link"] == 1]["event_code"].isin(
            ["OVERLOAD", "DRIFT_DETECTED", "NO_SIGNAL", "DEVICE_OFFLINE", "MODE_CHANGE", "SETPOINT_CHANGE"]
        ).sum()
    )

    summary_rows.append(
        {
            "building_id": building_name,
            "rows": n_total,
            "anomaly_events": n_anom,
            "anomaly_rate": anomaly_rate,
            "burst_like_anomaly_events": n_burst_events,
        }
    )
    print(f"Saved -> {save_path} ({n_total} rows, anomaly_rate={anomaly_rate:.3f})")

summary_df = pd.DataFrame(summary_rows).sort_values("building_id").reset_index(drop=True)
print("\nPer-building anomaly summary:")
display(summary_df)

max_rate = float(summary_df["anomaly_rate"].max()) if len(summary_df) else 0.0
print(f"Max anomaly rate across buildings: {max_rate:.3f}")
if max_rate > GEN_CONFIG["max_anomaly_rate"]:
    print("WARNING: anomaly rate exceeded configured max_anomaly_rate.")

Found sensor files: ['Office_Annika_anomalies.csv', 'Office_Cristina_anomalies.csv', 'Office_Jesus_anomalies.csv', 'PrimClass_Jaylin_anomalies.csv', 'PrimClass_Jolie_anomalies.csv']

Processing Office_Annika_anomalies.csv...
Saved → D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\logs_raw\Office_Annika_logs.csv (8688 rows)

Processing Office_Cristina_anomalies.csv...
Saved → D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\logs_raw\Office_Cristina_logs.csv (8703 rows)

Processing Office_Jesus_anomalies.csv...
Saved → D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\logs_raw\Office_Jesus_logs.csv (6843 rows)

Processing PrimClass_Jaylin_anomalies.csv...
Saved → D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\logs_raw\PrimClass_Jaylin_logs.csv (6702 rows)

Processing PrimClass_Jolie_anomalies.csv...
Saved → D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\logs_raw\PrimClass_Jolie_logs.csv (6716 ro

## Inspect one generated log file

Uses the first `*_logs.csv` found (no hardcoded building name).

In [25]:
log_files = sorted(f for f in os.listdir(LOGS_RAW) if f.endswith("_logs.csv"))
if not log_files:
    print("No log files yet.")
else:
    sample = os.path.join(LOGS_RAW, log_files[0])
    df = pd.read_csv(sample)
    print("Sample file:", sample)
    display(df.head())
    print(df["event_type"].value_counts())
    print(df["severity"].value_counts())
    print(df["anomaly_link"].value_counts())

Sample file: D:\Uni\Sem 4\PBL 1\smart-building-anomaly-detection\data\processed\logs_raw\Office_Annika_logs.csv


,timestamp,building_id,device_id,subsystem,event_type,event_code,severity,value,message,anomaly_link
0,2015-01-08 23:00:00,Office_Annika,AHU_1,HVAC,SYSTEM,DEVICE_OFFLINE,CRITICAL,25.781216,DEVICE_OFFLINE occurred,0
1,2015-01-09 00:00:00,Office_Annika,AHU_1,HVAC,CONTROL,SETPOINT_CHANGE,INFO,28.073766,SETPOINT_CHANGE occurred,0
2,2015-01-09 00:00:00,Office_Annika,AHU_2,HVAC,FAULT,OVERLOAD,CRITICAL,26.954752,OVERLOAD occurred,0
3,2015-01-09 00:55:00,Office_Annika,AHU_1,HVAC,SENSOR,CALIBRATION_WARNING,WARN,26.801253,CALIBRATION_WARNING occurred,1
4,2015-01-09 01:00:00,Office_Annika,AHU_1,HVAC,SENSOR,DRIFT_DETECTED,WARN,25.192031,DRIFT_DETECTED occurred,1


event_type
SYSTEM     3383
CONTROL    3102
SENSOR     1778
FAULT       425
Name: count, dtype: int64
severity
CRITICAL    3402
INFO        3093
WARN        2193
Name: count, dtype: int64
anomaly_link
0    6005
1    2683
Name: count, dtype: int64
